# GCM BA - 4. Verification of F1

On the EC-Earth3 grid, over the **LISFLOOD domain** (mask `ecearth3` of notebook 00c, no buffer; F1 itself is computed
over `REGION`, the RCM and EURO-CORDEX domains + 1 deg), with the statistics of every dataset (cached):

1. **integrity** of the adjusted runs;
2. **calibration**: EC-Earth3 historical + F1 against ERA5 and against raw EC-Earth3 (bias maps, scores, distributions,
   dry days, extremes), **pixel by pixel** (monthly quantiles) and **at the changes of month**. Same period as the training: QDM maps each quantile, so the adjusted historical run should match
   ERA5 closely (in-sample check). EC-Earth3 and ERA5 are not synchronised day by day: against ERA5 only the RMSE
   components that need no pairing are given (bias, spatial variability); RMSE and correlation for adjusted vs raw;
3. **signals preserved**: climate change (2025 / 2050 original - historical) and hosing (hosing - original), raw vs
   adjusted: maps, pixel-by-pixel regression (slope 1 = preserved), annual cycle, extremes, dry days, quantiles.

4. **strong precipitation amplification**: where the multiplicative QDM multiplies wet days by more than 5.

Figures and tables: `outputs/` next to this notebook (not in git); statistics cache on EOS.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_CORE = cfg["variables"]                                   # pr, tasmax, tasmin: required
VARS_ALL = VARS_CORE                # rsds not bias-adjusted (to add it back: VARS_CORE + cfg.get("extra_variables", []))
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]

CAL_PERIOD = ("1981", "2014")      # years of ERA5 / EC-Earth3 historical used for F1 (EC-Earth3 historical ends in 2014)
RSDS_FIRST_YEAR = "2005"           # ERA5 radiation starts in 2005: shorter period for rsds

def period_of(v):
    """Calibration period of a variable: CAL_PERIOD, starting in 2005 for rsds."""
    return (max(CAL_PERIOD[0], RSDS_FIRST_YEAR), CAL_PERIOD[1]) if v == "rsds" else CAL_PERIOD

REGION_MARGIN = 1.0                # degrees added around the RCM and EURO-CORDEX domains
REGION = bu.gcm_ba_region(cfg, REGION_MARGIN)   # Europe: box of the RCM and EURO-CORDEX domains (None = globe)
F1_TAG = (f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}"   # F1 version (ba_utils: CAL_WINDOW_MONTHS, PR_METHOD)
          + ("_europe" if REGION else ""))
print("region:", REGION)
import validation_utils as vu
OVERWRITE_CACHE = False
DOMAIN = vu.init(cfg, "domain", borders_res="01M")       # LISFLOOD domain (ecearth3 mask): statistics and maps over land
OUTF1 = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")    # bias-adjusted GCM runs (notebook 3)
CACHE = bu.gcm_ba_dir(cfg, "4_verification", "cache" + ("_europe" if REGION else ""))
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
LABEL = {"gcm_historical": "historical", "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
         "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical"),
         ("gcm_2050_nohosing", "gcm_historical", "2050 original - historical"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]
print("adjusted runs:", OUTF1, "| figures:", OUT)

## 0. Statistics (computed once, cached)

In [ ]:
def inputs(name):
    """Raw inputs (EC-Earth3 grid), standardised when read; ERA5 (CDO) and historical over the calibration period."""
    cal = name in ("era5", "gcm_historical")
    return {v: bu.gcm_ba_input(cfg, name, v, period_of(v) if cal else None, REGION) for v in VARS_ALL
            if bu.gcm_ba_available(cfg, name, v)}

def adjusted(name):
    f = OUTF1 / f"{name}_{F1_TAG}.nc"
    if not f.exists():
        return {}
    ds = xr.open_dataset(f, chunks={"time": 31}, **bu._TIME_KW)
    return {v: ds[v] for v in VARS_ALL if v in ds}

S_era5 = vu.compute_stats(inputs("era5"), CACHE / "era5_stats.nc", OVERWRITE_CACHE, label="ERA5 (EC-Earth3 grid)")
RAW, ADJ = {}, {}
for name, lab in LABEL.items():
    a = adjusted(name)
    if not a:
        print(f"[missing] {lab}: run notebook 3"); continue
    RAW[name] = vu.compute_stats(inputs(name), CACHE / f"{name}_raw_stats.nc", OVERWRITE_CACHE, label=f"{lab} raw")
    ADJ[name] = vu.compute_stats(a, CACHE / f"{name}_{F1_TAG}_stats.nc", OVERWRITE_CACHE, label=f"{lab} + F1")
VPLOT = [v for v in ("pr", "tasmax", "tasmin", "dtr", "rsds") if vu.has(S_era5, v) or any(vu.has(s, v) for s in ADJ.values())]
PAIRS = [p for p in PAIRS if p[0] in ADJ and p[1] in ADJ]
print("runs:", [LABEL[n] for n in ADJ], "| variables:", VPLOT)

## 1. Integrity of the adjusted runs

In [ ]:
rows = []
for name, s in ADJ.items():
    for v in [x for x in VARS_ALL if f"period_{x}" in s.attrs]:
        rows.append({"run": LABEL[name], "var": v, "period": s.attrs[f"period_{v}"], "n days": s.attrs[f"ndays_{v}"],
                     "min": round(s.attrs[f"min_{v}"], 3), "max": round(s.attrs[f"max_{v}"], 3)})
    rows.append({"run": LABEL[name], "var": "tasmin > tasmax (frac)", "period": "", "n days": "",
                 "min": s.attrs.get("frac_tasmin_gt_tasmax"), "max": ""})
display(pd.DataFrame(rows).set_index(["run", "var"]))

## 2. Calibration: EC-Earth3 historical + F1 vs ERA5 (in-sample)

### 2.1 Bias against ERA5: raw vs adjusted (pr in %; dots: |Welch t| > 2 on the yearly values)

In [ ]:
H_raw, H_adj = RAW.get("gcm_historical"), ADJ.get("gcm_historical")
SETS = {"ERA5": S_era5, "raw": H_raw, "F1": H_adj}
VCAL = [v for v in VPLOT if all(vu.has(s, v) for s in SETS.values())]
for v in VCAL:
    fig, axes = vu.map_axes(2, 3)
    lim = None
    for r, lab in enumerate(("raw", "F1")):
        sig = vu.welch_sig(vu.yearly(SETS[lab], v), vu.yearly(S_era5, v))
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            d = vu.diff(vu.seas(SETS[lab], v, s), vu.seas(S_era5, v, s), v)
            if lim is None:
                lim = float(np.nanpercentile(np.abs(d.where(vu.mask_for(d))), 98)) or 1.0
            vu.draw(axes[r, c], d, f"historical {lab} - ERA5: {v} {s}", cmap=vu.DCMAP[v], sym=True, lim=lim,
                    sig=sig if s == "ANN" else None, label=vu.dlabel(v))
    plt.tight_layout(); plt.savefig(OUT / f"calibration_bias_{v}.png", dpi=110); plt.show()

### 2.2 Scores (climatologies, over the domain)

In [ ]:
rows_e, rows_r = [], []
for v in VCAL:
    for s in ("ANN", "DJF", "MAM", "JJA", "SON"):
        ref, raw_, adj_ = vu.seas(S_era5, v, s), vu.seas(H_raw, v, s), vu.seas(H_adj, v, s)
        for lab, x in (("raw", raw_), ("F1", adj_)):
            rows_e.append({"var": v, "season": s, "dataset": lab, **{k: round(val, 3) for k, val in vu.scores_no_pairing(x, ref).items()}})
        sc = vu.scores(adj_, raw_)
        rows_r.append({"var": v, "season": s, **{k: round(sc[k], 3) for k in ("bias", "RMSE", "corr", "sd ratio")}})
t1 = pd.DataFrame(rows_e).pivot_table(index=["var", "season"], columns="dataset", values=["bias", "sd ratio", "sd diff"])
t1.to_csv(OUT / "calibration_scores_vs_era5.csv"); print("Against ERA5 (bias, spatial variability):"); display(t1)
t2 = pd.DataFrame(rows_r).set_index(["var", "season"])
t2.to_csv(OUT / "calibration_scores_adjusted_vs_raw.csv"); print("Adjusted against raw:"); display(t2)

### 2.3 Distributions (Q-Q against ERA5), dry days, extremes

In [ ]:
keys = [k for k in ("pr_q_wet", "tasmax_q", "tasmin_q", "dtr_q", "rsds_q") if all(k in s for s in SETS.values())]
fig, axes = plt.subplots(1, len(keys), figsize=(4.5 * len(keys), 4.2), squeeze=False)
for ax, k in zip(axes[0], keys):
    ref = S_era5[k].values
    for lab, col in (("raw", "k"), ("F1", "tab:red")):
        ax.plot(ref, SETS[lab][k].values, "o-", ms=3, color=col, label=lab)
    lo, hi = np.nanmin(ref), np.nanmax(ref)
    ax.plot([lo, hi], [lo, hi], color="grey", lw=0.8, ls="--")
    ax.set_xlabel("ERA5 quantile"); ax.set_ylabel("EC-Earth3 historical quantile")
    ax.set_title(k.replace("_q_wet", " (wet days)").replace("_q", ""))
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "calibration_qq.png", dpi=110); plt.show()
print("pooled dry-day fraction:", {lab: round(s.attrs.get("pooled_dry_frac", np.nan), 3) for lab, s in SETS.items()})

rows = []
for kind, name in (("dry", "dry-day fraction"), ("sdii", "wet-day intensity")):
    if all(f"pr_year_{kind}" in s for s in SETS.values()):
        for lab in ("raw", "F1"):
            rows.append({"index": name, "comparison": f"{lab} vs ERA5", **{k: round(x, 3) for k, x in vu.scores_no_pairing(vu.seas(SETS[lab], "pr", "ANN", kind), vu.seas(S_era5, "pr", "ANN", kind)).items()}})
for v, (how, name) in vu.EXT.items():
    if all(f"{v}_year_ext" in s for s in SETS.values()):
        for lab in ("raw", "F1"):
            rows.append({"index": name, "comparison": f"{lab} vs ERA5", **{k: round(x, 3) for k, x in vu.scores_no_pairing(vu.seas(SETS[lab], v, "ANN", "ext"), vu.seas(S_era5, v, "ANN", "ext")).items()}})
display(pd.DataFrame(rows).set_index(["index", "comparison"]))

### 2.4 Distributions pixel by pixel (the QDM property)

QDM is trained per pixel and per calendar month: over the calibration period the adjusted historical run should
reproduce, **at every pixel and for every month**, the quantiles of ERA5 (pr: wet-day intensities, >= 0.1 mm/day).
Maps: difference with ERA5 of the median, 90th and 99th percentiles, averaged over the 12 months (pr in %), raw (top)
and adjusted (bottom). Table: mean absolute difference over the domain (all months), raw and adjusted.

In [ ]:
QS = (0.5, 0.9, 0.99)
VQ = [v for v in ("pr", "tasmax", "tasmin") if v in VARS_ALL]
Q = {"ERA5": vu.monthly_quantiles({v: inputs("era5")[v] for v in VQ}, CACHE / "era5_monthly_quantiles.nc",
                                  OVERWRITE_CACHE, QS, label="ERA5"),
     "raw": vu.monthly_quantiles({v: inputs("gcm_historical")[v] for v in VQ}, CACHE / "gcm_historical_raw_monthly_quantiles.nc",
                                 OVERWRITE_CACHE, QS, label="historical raw"),
     "F1": vu.monthly_quantiles({v: adjusted("gcm_historical")[v] for v in VQ},
                                CACHE / f"gcm_historical_{F1_TAG}_monthly_quantiles.nc", OVERWRITE_CACHE, QS, label="historical + F1")}


def qdiff(x, ref, v):
    """x - ref per month and quantile (pr: in % of ref, where ref >= 0.1 mm/day)."""
    return (100 * (x - ref) / ref).where(ref >= bu.THRESH_PR) if v == "pr" else x - ref


rows = []
for v in VQ:
    fig, axes = vu.map_axes(2, len(QS))
    lim = None
    for r, lab in enumerate(("raw", "F1")):
        d = qdiff(Q[lab][v], Q["ERA5"][v], v)
        for c, q in enumerate(QS):
            dq = d.sel(quantile=q)
            m = dq.mean("month")
            if lim is None:
                lim = float(np.nanpercentile(np.abs(m.where(vu.mask_for(m))), 98)) or 1.0
            vu.draw(axes[r, c], m, f"historical {lab} - ERA5: {v} p{int(100 * q)}", cmap=vu.DCMAP[v], sym=True, lim=lim,
                    label=vu.dlabel(v))
            rows.append({"var": v, "quantile": f"p{int(100 * q)}", "series": lab,
                         "mean |diff| (all months)": vu.land_stats(np.abs(dq).mean("month"))[0]})
    plt.tight_layout(); plt.savefig(OUT / f"calibration_pixel_quantiles_{v}.png", dpi=110); plt.show()
pq = pd.DataFrame(rows).pivot_table(index=["var", "quantile"], columns="series", values="mean |diff| (all months)").round(3)
pq.to_csv(OUT / "calibration_pixel_quantiles.csv")
print("Mean absolute difference with ERA5 over the domain (pr in %, temperatures in degC): should drop to ~0 with F1")
display(pq)

### 2.5 Changes of month (each month keeps its own correction)

F1 is calibrated and applied month by month, without blending consecutive months, so the correction can jump at a
change of month. Daily climatology (1981-2014) of the correction (adjusted - raw, historical run); step at each change
of month (mean of the 5 first days of the new month - mean of the 5 last days of the old one), compared with the same
between two 5-day blocks **inside** the months (natural evolution + noise). Steps at the changes of month much larger
than inside the months would be a visible artefact.

In [ ]:
VJ = [v for v in ("tasmax", "tasmin", "pr") if v in VARS_ALL]
C_raw = vu.daily_climatology({v: inputs("gcm_historical")[v] for v in VJ}, CACHE / "gcm_historical_raw_daily_clim.nc",
                             OVERWRITE_CACHE, label="historical raw")
C_adj = vu.daily_climatology({v: adjusted("gcm_historical")[v] for v in VJ}, CACHE / f"gcm_historical_{F1_TAG}_daily_clim.nc",
                             OVERWRITE_CACHE, label="historical + F1")
rows, steps = [], {}
for v in VJ:
    corr = C_adj[v] - C_raw[v]                                  # correction, mm/day for pr
    at, inside = vu.month_steps(corr)
    steps[v] = np.abs(at).max("boundary")
    a_mean, a_p95 = vu.land_stats(np.abs(at).mean("boundary"))
    i_mean, i_p95 = vu.land_stats(np.abs(inside).mean("block"))
    rows.append({"var": v, "unit": vu.UNITS[v], "|step| at changes of month: mean": a_mean, "p95": a_p95,
                 "|step| inside months: mean": i_mean, "p95 ": i_p95, "ratio of the means": a_mean / i_mean})
jumps = pd.DataFrame(rows).set_index("var").round(3)
jumps.to_csv(OUT / "month_transitions.csv"); display(jumps)

fig, axes = plt.subplots(1, len(VJ), figsize=(6 * len(VJ), 3.2), squeeze=False)
for ax, v in zip(axes[0], VJ):
    corr = vu.dmean(C_adj[v] - C_raw[v])
    ax.plot(corr.dayofyear, corr, lw=1)
    for s in vu.MONTH_START:
        ax.axvline(s + 1, color="grey", lw=0.5, ls=":")
    ax.set_title(f"correction adjusted - raw over the domain: {v} ({vu.UNITS[v]})", fontsize=9); ax.set_xlabel("day of the year")
plt.tight_layout(); plt.savefig(OUT / "month_transitions_cycle.png", dpi=110); plt.show()

fig, axes = vu.map_axes(1, len(VJ))
for ax, v in zip(axes[0], VJ):
    vu.draw(ax, steps[v], f"largest |step| at a change of month: {v}", cmap="magma_r", label=vu.UNITS[v],
            lim=(0, float(np.nanpercentile(steps[v].where(vu.mask_for(steps[v])), 98)) or 1.0))
plt.tight_layout(); plt.savefig(OUT / "month_transitions_maps.png", dpi=110); plt.show()

## 3. Signals preserved? raw vs adjusted deltas (same grid)

### 3.1 Delta maps (pr in %): raw, adjusted, difference

In [ ]:
def delta(a, b, v, season="ANN", kind="mean"):
    return vu.diff(vu.seas(a, v, season, kind), vu.seas(b, v, season, kind), v)

for a, b, lab in PAIRS:
    vs = [v for v in VPLOT if all(vu.has(s, v) for s in (RAW[a], RAW[b], ADJ[a], ADJ[b]))]
    fig, axes = vu.map_axes(len(vs), 3)
    for r, v in enumerate(vs):
        d_raw, d_adj = delta(RAW[a], RAW[b], v), delta(ADJ[a], ADJ[b], v)
        lim = float(np.nanpercentile(np.abs(d_raw.where(vu.mask_for(d_raw))), 98)) or 1.0
        vu.draw(axes[r, 0], d_raw, f"raw: {v}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v),
                sig=vu.welch_sig(vu.yearly(RAW[a], v), vu.yearly(RAW[b], v)))
        vu.draw(axes[r, 1], d_adj, f"F1: {v}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v),
                sig=vu.welch_sig(vu.yearly(ADJ[a], v), vu.yearly(ADJ[b], v)))
        vu.draw(axes[r, 2], d_adj - d_raw, f"F1 - raw delta: {v}", cmap="PiYG", sym=True, lim=lim, label=vu.dlabel(v))
    fig.suptitle(lab, y=1.0)
    plt.tight_layout(); plt.savefig(OUT / f"delta_{a}_minus_{b}.png", dpi=110, bbox_inches="tight"); plt.show()

### 3.2 Is the delta preserved? Regression of the adjusted delta on the raw delta, pixel by pixel

Preserved: slope ~ 1, corr ~ 1, mean ratio ~ 1. QDM preserves the change quantile by quantile by construction
(exactly for additive variables); the fixed WDF may modify the dry-day part of the precipitation change.

In [ ]:
rows = []
INDICES = [(v, "mean", v) for v in VPLOT] + [(v, "ext", vu.EXT[v][1]) for v in vu.EXT] + \
          [("pr", "dry", "dry-day fraction"), ("pr", "sdii", "wet-day intensity")]
for a, b, lab in PAIRS:
    for v, kind, name in INDICES:
        key = {"mean": f"{v}_year_mean", "ext": f"{v}_year_ext", "dry": "pr_year_dry", "sdii": "pr_year_sdii"}[kind]
        if not all(key in s for s in (RAW[a], RAW[b], ADJ[a], ADJ[b])):
            continue
        for season in (("ANN", "DJF", "JJA") if kind == "mean" else ("ANN",)):
            if kind == "dry":
                d_raw = 100 * (vu.seas(RAW[a], v, season, kind) - vu.seas(RAW[b], v, season, kind))
                d_adj = 100 * (vu.seas(ADJ[a], v, season, kind) - vu.seas(ADJ[b], v, season, kind))
            else:
                d_raw, d_adj = delta(RAW[a], RAW[b], v, season, kind), delta(ADJ[a], ADJ[b], v, season, kind)
            rows.append({"pair": lab, "index": name, "season": season, **{k: round(x, 3) for k, x in vu.regression(d_raw, d_adj).items()}})
pres = pd.DataFrame(rows).set_index(["pair", "index", "season"])
pres.to_csv(OUT / "delta_preservation.csv"); display(pres)

### 3.2b Precipitation signals by climate zone and over Europe

The pr changes of 3.2 are relative (%) per pixel: in dry regions they are large and noisy, and the bias adjustment
changes their base, which can lower the slope over the whole domain even where the humid regions are well preserved.
Here, per climate zone (ERA5 1981-2014 mean pr) and over Europe (35-72 N, 25 W - 45 E):
1. **pattern**: slope / correlation of the adjusted change on the raw change (as in 3.2), with the dry-day fraction
   also as a **relative** change (%), which the fixed WDF should preserve better than the change in points;
2. **regional mean**: change of the regional mean pr (%), raw and adjusted, and their ratio (1 = preserved).

In [ ]:
g = next(iter(RAW.values()))
clim = vu.seas(S_era5, "pr", "ANN").assign_coords(lat=g.lat, lon=g.lon)
ZONES = {"arid (< 0.5 mm/day)": clim < 0.5, "semi-arid (0.5 - 2)": (clim >= 0.5) & (clim < 2), "humid (>= 2)": clim >= 2,
         "Europe": (clim.lat >= 35) & (clim.lat <= 72) & (clim.lon >= -25) & (clim.lon <= 45)}


def change(src, a, b, season, kind):
    """Change a - b of a pr index (%); 'dry_rel' = relative change of the dry-day fraction."""
    if kind == "dry_rel":
        x, y = vu.seas(src[a], "pr", season, "dry"), vu.seas(src[b], "pr", season, "dry")
        return (100 * (x - y) / y).where(y > 0.01)
    return delta(src[a], src[b], "pr", season, kind)


ITEMS = [("mean", "ANN", "pr"), ("mean", "DJF", "pr"), ("mean", "JJA", "pr"), ("ext", "ANN", "Rx1day"),
         ("sdii", "ANN", "wet-day intensity"), ("dry_rel", "ANN", "dry-day fraction (relative)")]
rows, reg = [], []
for a, b, lab in PAIRS:
    if not all(vu.has(s, "pr") for s in (RAW[a], RAW[b], ADJ[a], ADJ[b])):
        continue
    for kind, season, name in ITEMS:
        d_raw, d_adj = change(RAW, a, b, season, kind), change(ADJ, a, b, season, kind)
        for z, m in ZONES.items():
            if int((m & vu.mask_for(d_raw) & d_raw.notnull() & d_adj.notnull()).sum()) < 10:
                continue                                         # (almost) no pixel of this zone in the domain
            r = vu.regression(d_raw.where(m), d_adj.where(m))
            rows.append({"pair": lab, "index": name, "season": season, "zone": z, "slope": r["slope"], "corr": r["corr"]})
    for season in ("ANN", "DJF", "JJA"):
        for z, m in ZONES.items():
            ch = {}
            for src, k in ((RAW, "raw (%)"), (ADJ, "adjusted (%)")):
                x, y = vu.dmean(vu.seas(src[a], "pr", season).where(m)), vu.dmean(vu.seas(src[b], "pr", season).where(m))
                ch[k] = float(100 * (x - y) / y)
            reg.append({"pair": lab, "season": season, "zone": z, **ch, "ratio": ch["adjusted (%)"] / ch["raw (%)"]})

zt = pd.DataFrame(rows).pivot_table(index=["pair", "index", "season"], columns="zone", values="slope", sort=False)
zt = zt[[z for z in ZONES if z in zt.columns]].round(2)
zt.to_csv(OUT / "delta_preservation_pr_zones.csv")
zc = pd.DataFrame(rows).pivot_table(index=["pair", "index", "season"], columns="zone", values="corr", sort=False)
zc = zc[[z for z in ZONES if z in zc.columns]].round(2)
zc.to_csv(OUT / "delta_preservation_pr_zones_corr.csv")
rt = pd.DataFrame(reg).set_index(["pair", "season", "zone"]).round(3)
rt.to_csv(OUT / "delta_preservation_pr_regional_means.csv")
with pd.option_context("display.max_rows", 500):
    print("1. Pattern: slope of the adjusted change on the raw change, per zone (1 = preserved)"); display(zt)
    print("   correlation"); display(zc)
    print("2. Change of the regional mean pr (%), raw and adjusted"); display(rt)

### 3.3 Annual cycle of the deltas over the domain, and deltas quantile by quantile

In [ ]:
fig, axes = plt.subplots(len(PAIRS), len(VPLOT), figsize=(4.4 * len(VPLOT), 3.1 * len(PAIRS)), squeeze=False)
for r, (a, b, lab) in enumerate(PAIRS):
    for c, v in enumerate(VPLOT):
        ax = axes[r, c]
        if not all(vu.has(s, v, "clim_month") for s in (RAW[a], RAW[b], ADJ[a], ADJ[b])):
            ax.set_visible(False); continue
        for src, col, nm in ((RAW, "k", "raw"), (ADJ, "tab:red", "F1")):
            ma, mb = vu.dmean(src[a][f"{v}_clim_month"]), vu.dmean(src[b][f"{v}_clim_month"])
            d = 100 * (ma - mb) / mb if v == "pr" else ma - mb
            ax.plot(d.month, d, "o-", ms=3, color=col, label=nm)
        ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{lab}: {v} ({vu.dlabel(v)})", fontsize=8)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "delta_annual_cycle.png", dpi=110); plt.show()

keys = ["pr_q_wet", "tasmax_q", "tasmin_q", "dtr_q", "rsds_q"]
fig, axes = plt.subplots(len(PAIRS), len(keys), figsize=(4.2 * len(keys), 3.1 * len(PAIRS)), squeeze=False)
for r, (a, b, lab) in enumerate(PAIRS):
    for c, k in enumerate(keys):
        ax = axes[r, c]
        if not all(k in s for s in (RAW[a], RAW[b], ADJ[a], ADJ[b])):
            ax.set_visible(False); continue
        rel = k.startswith("pr")
        for src, col, nm, mk in ((RAW, "k", "raw", "o-"), (ADJ, "tab:red", "F1", "s--")):
            d = (100 * (src[a][k] - src[b][k]) / src[b][k]) if rel else (src[a][k] - src[b][k])
            ax.plot(vu.QUANTILES, d.values, mk, ms=3, color=col, label=nm)
        ax.set_xscale("logit"); ax.axhline(0, color="grey", lw=0.5)
        ax.set_title(f"{lab}: {k.replace('_q_wet', ' wet').replace('_q', '')} ({'%' if rel else 'abs.'})", fontsize=8)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "delta_quantiles.png", dpi=110); plt.show()

## 4. Strong precipitation amplification

Multiplicative QDM can multiply a wet day by a large factor where the model is much drier than ERA5 at a given
quantile. Per pixel, over the wet days of the raw run (>= 1 mm/day): share of days multiplied by more than 5, largest
factor; table by run and by climate of the pixel (ERA5 1981-2014 mean pr), and the pixels with the largest factors.

In [ ]:
AMP = {}
for name, lab in LABEL.items():
    a = adjusted(name)
    if "pr" not in a:
        continue
    raw = bu.gcm_ba_input(cfg, name, "pr", period_of("pr") if name == "gcm_historical" else None, REGION)
    AMP[name] = vu.pr_amplification(a["pr"], raw, CACHE / f"{name}_{F1_TAG}_pr_amplification.nc", OVERWRITE_CACHE, label=lab)

clim = vu.seas(S_era5, "pr", "ANN").assign_coords(lat=AMP["gcm_historical"].lat, lon=AMP["gcm_historical"].lon)
zones = {"arid (< 0.5 mm/day)": clim < 0.5, "semi-arid (0.5 - 2)": (clim >= 0.5) & (clim < 2), "humid (>= 2)": clim >= 2}
rows = []
for name, s in AMP.items():
    row = {"run": LABEL[name], "share of wet days x5 or more (%)": vu.land_stats(s["frac_big"])[0],
           "largest factor": float(s["max_factor"].where(vu.mask_for(s["max_factor"])).max()),
           "largest adjusted pr (mm/day)": float(s["adj_max"].where(vu.mask_for(s["adj_max"])).max()),
           "largest raw pr (mm/day)": float(s["raw_max"].where(vu.mask_for(s["raw_max"])).max())}
    for z, m in zones.items():
        row[f"x5 share, {z} (%)"] = float(vu.dmean(s["frac_big"].where(m)))
    rows.append(row)
amp = pd.DataFrame(rows).set_index("run").round(3)
amp.to_csv(OUT / "pr_amplification.csv"); display(amp)

h = AMP["gcm_historical"]
fig, axes = vu.map_axes(1, 2)
vu.draw(axes[0, 0], h["frac_big"], "historical: share of wet days multiplied by > 5 (%)", cmap="magma_r", label="%",
        lim=(0, float(np.nanpercentile(h["frac_big"].where(vu.mask_for(h["frac_big"])), 99)) or 1.0))
vu.draw(axes[0, 1], h["max_factor"], "historical: largest factor", cmap="magma_r", label="factor", lim=(1, 20))
plt.tight_layout(); plt.savefig(OUT / "pr_amplification_maps.png", dpi=110); plt.show()

top = h.where(vu.mask_for(h["max_factor"])).to_dataframe().dropna().sort_values("max_factor", ascending=False).head(10)
top["ERA5 mean pr (mm/day)"] = [float(clim.sel(lat=la, lon=lo)) for la, lo in top.index]
print("Historical: the 10 pixels with the largest factors"); display(top.round(2))

### 4b. Order of the wet days around the model's last historical quantile

Above the model's last historical quantile hq (per pixel and month), pr gets the factor of the last quantile (plain
QDM, notebook 3 `PR_TAIL = None`; with `PR_TAIL = "additive"`, QDM(hq) + (x - hq) instead). QDM takes its factor from
the rank of each day, so it can swap two nearly equal days. Check: per pixel and month, the wet days
are sorted by their raw value; an **inversion** is a day that ends lower than a day with a smaller raw value (drop >
0.01 mm/day). Counted **below**, **across** and **above** hq (with the additive rule, "across" = the switch of rule); share of the pixel-months
with at least one inversion, inverted pairs per million pairs compared, largest drop. Land and Europe.

In [ ]:
F1_DIR = bu.gcm_ba_dir(cfg, "2_F1_calculation", F1_TAG)
HQ = xr.open_dataset(F1_DIR / "qdm_pr.nc")["hist_q"].isel(quantiles=-1, drop=True).squeeze(drop=True).load()
in_europe = lambda d: (d.lat >= 35) & (d.lat <= 72) & (d.lon >= -25) & (d.lon <= 45)
ORD, rows = {}, []
for name, lab in LABEL.items():
    a = adjusted(name)
    if "pr" not in a:
        continue
    raw = bu.gcm_ba_input(cfg, name, "pr", period_of("pr") if name == "gcm_historical" else None, REGION)
    o = vu.pr_order(a["pr"], raw, HQ.assign_coords(lat=raw.lat, lon=raw.lon), CACHE / f"{name}_{F1_TAG}_pr_order.nc",
                    OVERWRITE_CACHE, label=lab)
    ORD[name] = o
    land = vu.mask_for(o["n_pairs"]) & (o["n_pairs"] > 0)
    for zone, z in (("land", land), ("Europe", land & in_europe(o))):
        s = lambda v: o[v].where(z)
        n_pm, n_pairs = int(z.sum()), float(s("n_pairs").sum())
        n_inv = float((s("inv_below") + s("inv_across") + s("inv_above")).sum())
        rows.append({"run": lab, "zone": zone,
                     "pixel-months with an inversion: across hq (%)": 100 * int((s("inv_across") > 0).sum()) / n_pm,
                     "below hq (%)": 100 * int((s("inv_below") > 0).sum()) / n_pm,
                     "above hq (%)": 100 * int((s("inv_above") > 0).sum()) / n_pm,
                     "inverted pairs (per million)": 1e6 * n_inv / n_pairs,
                     "largest drop across hq (mm/day)": float(s("max_drop_across").max()),
                     "largest drop (mm/day)": float(s("max_drop").max()),
                     "wet days above hq (%)": 100 * float(s("n_above").sum()) / (n_pairs + n_pm)})
order = pd.DataFrame(rows).set_index(["run", "zone"]).round(3)
order.to_csv(OUT / "pr_order_around_hq.csv"); display(order)

worst = max(ORD, key=lambda n: float(ORD[n]["max_drop_across"].where(vu.mask_for(ORD[n]["n_pairs"])).max()))
fig, axes = vu.map_axes(1, 2)
d = ORD[worst]["max_drop_across"].max("month")
vu.draw(axes[0, 0], d, f"{LABEL[worst]}: largest drop across hq (mm/day)", cmap="magma_r", label="mm/day",
        lim=(0, float(np.nanpercentile(d.where(vu.mask_for(d) & (d > 0)), 99)) or 1.0))
n = (ORD[worst]["inv_across"] > 0).sum("month")
vu.draw(axes[0, 1], n, f"{LABEL[worst]}: months with an inversion across hq", cmap="magma_r", label="months", lim=(0, 12))
plt.tight_layout(); plt.savefig(OUT / "pr_order_around_hq.png", dpi=110); plt.show()

## Conclusions (to fill in)

- Calibration (historical + F1 vs ERA5): ...
- Climate-change and hosing deltas preserved? (slope, mean ratio; dry days): ...
- Points to discuss before the RCM bias adjustment: ...